# AIC HCMC 2026 — chạy SOTUYEN2 bằng Zilliz

Notebook này dùng ba collection `aic_visual_mobileclip_s2_v1`, `aic_text_mobileclip_s2_v1`, `aic_object_detection_v1` và file canonical `metadata.parquet`. Visual/text query được mã hoá bằng đúng `MobileCLIP-S2` + `datacompdr`.

Trước khi chạy, vào **Colab → Secrets** (biểu tượng chìa khoá), tạo hai secret và bật **Notebook access**:

- `ZILLIZ_URI`: Public Endpoint của cluster, ví dụ `https://...api.zillizcloud.com`.
- `ZILLIZ_TOKEN`: API key/token của cùng cluster.

Không điền token vào code, GitHub hoặc output notebook. Object metadata hiện còn thiếu một số nhóm video không làm mất candidate: object chỉ là một phiếu RRF bổ sung, không hard-filter kết quả.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Lấy code mới nhất từ branch dev

Code được clone sạch vào ổ tạm `/content`; notebook không dùng bản repo cũ đang nằm trong Drive. Mỗi lần Colab khởi động lại, chạy lại cell này để lấy `dev` mới nhất.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/Minhthien2103/AIC-HCMC-26.git'
REPO_DIR = Path('/content/AIC-HCMC-26')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', 'dev', '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
elif (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', 'dev'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'dev'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'dev'], check=True)
else:
    raise RuntimeError(f'{REPO_DIR} tồn tại nhưng không phải Git repository')

subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=True)

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)

## 2. Chỉ sửa bốn đường dẫn Drive ở cell dưới

Nếu folder được chia sẻ qua link, hãy **Add shortcut to Drive** hoặc upload nó vào My Drive trước. `METADATA_PATH` phải trỏ tới file `zilliz_exports/metadata.parquet` do notebook upload object tạo ra, không phải ba file scalar/mapping phụ.

In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')

QUERY_DIR = MY_DRIVE / 'SOTUYEN2-bo-de-thi'
METADATA_PATH = MY_DRIVE / 'extracted_metadata' / 'zilliz_exports' / 'metadata.parquet'
KEYFRAMES_DIR = MY_DRIVE / 'AIC_HCMC_26' / 'AIC-HCMC-26' / 'data' / 'keyframes'
OUTPUT_DIR = MY_DRIVE / 'AIC_HCMC_26' / 'sotuyen2_zilliz_output'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for label, path in {
    'QUERY_DIR': QUERY_DIR,
    'METADATA_PATH': METADATA_PATH,
    'KEYFRAMES_DIR': KEYFRAMES_DIR,
}.items():
    if not path.exists():
        raise FileNotFoundError(f'{label} không tồn tại: {path}. Hãy sửa đường dẫn ở cell này.')

query_files = list(QUERY_DIR.glob('*.txt'))
print('Query files:', len(query_files))
print('Metadata:', METADATA_PATH)
print('Keyframes:', KEYFRAMES_DIR)
if len(query_files) != 30:
    raise RuntimeError(f'SOTUYEN2 phải có 30 file .txt, hiện có {len(query_files)}')

## 3. Đọc URI và Token từ Colab Secrets

Cell chỉ báo đã nạp secret, không in giá trị ra màn hình.

In [ ]:
import os
from google.colab import userdata

def first_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value:
            return value
    return None

os.environ['ZILLIZ_URI'] = first_secret('ZILLIZ_URI', 'MILVUS_URI') or ''
os.environ['ZILLIZ_TOKEN'] = first_secret('ZILLIZ_TOKEN', 'MILVUS_TOKEN') or ''
if not os.environ['ZILLIZ_URI'] or not os.environ['ZILLIZ_TOKEN']:
    raise RuntimeError('Thiếu ZILLIZ_URI/ZILLIZ_TOKEN (hoặc MILVUS_URI/MILVUS_TOKEN) trong Colab Secrets')

os.environ['AIC_RETRIEVAL_BACKEND'] = 'zilliz'
os.environ['AIC_METADATA_PATH'] = str(METADATA_PATH)
os.environ['AIC_KEYFRAMES_DIR'] = str(KEYFRAMES_DIR)
print('Đã nạp ZILLIZ_URI và ZILLIZ_TOKEN từ Colab Secrets (không hiển thị giá trị).')

## 4. Preflight — bắt buộc phải PASSED

Bước này chưa tải MobileCLIP/Qwen. Nó kiểm tra kết nối, tên ba collection, search visual + subtitle, phép ánh xạ metadata, 20 đường dẫn keyframe mẫu và cấu trúc 30 query.

In [ ]:
validate_command = [
    sys.executable, str(REPO_DIR / 'scripts' / 'validate_zilliz_setup.py'),
    '--metadata-path', str(METADATA_PATH),
    '--keyframes-dir', str(KEYFRAMES_DIR),
    '--queries-dir', str(QUERY_DIR),
]
validation = subprocess.run(
    validate_command, cwd=REPO_DIR, text=True, capture_output=True
)
print(validation.stdout or '', end='')
if validation.stderr:
    print(validation.stderr, end='')
validation.check_returncode()

## 5. Chạy toàn bộ SOTUYEN2

Mặc định `4bit` phù hợp T4/L4. Nếu dùng A100 và đủ VRAM, đổi `--vlm-mode` thành `bf16`. Checkpoint nằm trên Drive; khi Colab ngắt, chạy lại notebook và cell này sẽ tiếp tục các query đã hoàn tất. Lần chạy đầu sẽ tải MobileCLIP, model dịch và Qwen2-VL.

In [ ]:
RESULT_ZIP = OUTPUT_DIR / 'SOTUYEN2_submission.zip'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
PROVENANCE_DIR = OUTPUT_DIR / 'provenance'

run_command = [
    sys.executable, str(REPO_DIR / 'scripts' / 'generate_submission.py'),
    '--queries-dir', str(QUERY_DIR),
    '--output', str(RESULT_ZIP),
    '--retrieval-backend', 'zilliz',
    '--metadata-path', str(METADATA_PATH),
    '--keyframes-dir', str(KEYFRAMES_DIR),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--provenance-dir', str(PROVENANCE_DIR),
    '--resume',
    '--device', 'cuda',
    '--vlm-mode', '4bit',
    '--max-rows', '100',
]
subprocess.run(run_command, cwd=REPO_DIR, check=True)
print('Hoàn tất:', RESULT_ZIP)

## Kết quả

- ZIP submission: `OUTPUT_DIR/SOTUYEN2_submission.zip`
- CSV checkpoint từng query: `OUTPUT_DIR/checkpoints/`
- Provenance, model/config và review candidates: `OUTPUT_DIR/provenance/`

Token Zilliz không được ghi vào bất kỳ file kết quả nào.